# Chapter 7 (Capstone): When Diffusion Runs Backwards

Companion notebook to Section 7 of the study guide. It combines the chain rule and characteristics (B.1), a change of
variables (B.2), a divergence and the Exner equation (B.3) and the Fourier view of the delta function (B.4), applied to the
high-angle-wave instability of sandy coastlines (Ashton et al. 2001; Ashton and Murray 2006).

Before committing: **Restart** the kernel, then **Run All**, then **save**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from ch07_tools import (ashton_flux, ashton_flux_derivative, blowup_time, cerc_flux,
                        cerc_flux_derivative, critical_angle, efolding_time, gaussian_bump,
                        run_one_line, shoreline_diffusivity)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"                   # categorical slots 1 and 2
RAMP = ["#86b6ef", "#2a78d6", "#104281"]              # three ordered series
RAMP4 = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]  # four ordered series
GRAY = "#52514e"                                      # reference curves

## 7.1 A one-line shoreline model

$x$ runs alongshore, $y$ offshore; the shoreline is $y_s(x,t)$ with orientation $\theta_s = \arctan(\partial y_s/\partial x)$.
Waves arrive with crests at $\alpha_0$ to the $x$-axis, so the relative angle is $\alpha = \alpha_0 - \theta_s$.
Sediment conservation over a shoreface of depth $D_{sf}$ is a one-dimensional Exner equation: $D_{sf}\,\dfrac{\partial y_s}{\partial t} = -\dfrac{\partial Q_s}{\partial x}$.

*My notes:*

## 7.2 The chain rule turns it into a diffusion equation

$\dfrac{\partial y_s}{\partial t} = \mathcal D\,\dfrac{\partial^2 y_s}{\partial x^2}$ with $\mathcal D = \dfrac{1}{D_{sf}}\,\dfrac{dQ_s/d\alpha}{1 + (\partial y_s/\partial x)^2}$.
For $Q_s = K\sin\alpha\cos\alpha$, $dQ_s/d\alpha = K\cos2\alpha$: positive below $45^\circ$, **negative** above.

*My notes:*

In [ ]:
# The chain-rule step, done by SymPy for a general flux law Q(alpha)
x, t, alpha0, D_sf = sp.symbols("x t alpha_0 D_sf", positive=True)
y = sp.Function("y")(x, t)
Q = sp.Function("Q")
alpha = alpha0 - sp.atan(sp.diff(y, x))              # alpha = alpha0 - theta_s
dydt = -sp.diff(Q(alpha), x) / D_sf                  # one-line Exner equation
D = sp.diff(Q(alpha), alpha0) / (D_sf * (1 + sp.diff(y, x) ** 2))   # dQ/dalpha at alpha
print(sp.simplify(dydt - D * sp.diff(y, x, 2)))      # 0

In [ ]:
alpha_deg = np.linspace(0, 89.5, 400)
a = np.radians(alpha_deg)
a_cerc, a_ashton = critical_angle(cerc_flux_derivative), critical_angle(ashton_flux_derivative)
print(f"diffusivity changes sign at {np.degrees(a_cerc):.2f} deg (sin cos) "
      f"and {np.degrees(a_ashton):.2f} deg (sin cos^(6/5))")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
q_ash_max = ashton_flux(a_ashton)
axes[0].plot(alpha_deg, cerc_flux(a) / cerc_flux(a_cerc), color=BLUE, label="sin a cos a")
axes[0].plot(alpha_deg, ashton_flux(a) / q_ash_max, "--", color=ORANGE, label="sin a cos^(6/5) a")
axes[0].axvspan(45, 90, color="#f0efec")
axes[0].text(70, 0.10, "anti-diffusive\n(unstable)", ha="center", fontsize=8, color=GRAY)
axes[0].text(25, 0.35, "diffusive\n(smoothing)", ha="center", fontsize=8, color=GRAY)
axes[0].set(xlim=(0, 90), ylim=(0, 1.1), xlabel="relative wave angle (degrees)",
            ylabel="Q_s / Q_s,max", title="(a) alongshore flux has a maximum")
axes[0].legend(fontsize=8, loc="upper left")

axes[1].plot(alpha_deg, cerc_flux_derivative(a), color=BLUE, label="sin a cos a")
axes[1].plot(alpha_deg, ashton_flux_derivative(a), "--", color=ORANGE, label="sin a cos^(6/5) a")
axes[1].axhline(0, color=GRAY, lw=0.8)
for ac in (a_ashton, a_cerc):
    axes[1].axvline(np.degrees(ac), color=GRAY, lw=0.8, ls=":")
axes[1].set(xlim=(0, 90), xlabel="relative wave angle (degrees)", ylabel="dQ_s / dalpha  (units of K)",
            title="(b) the diffusivity changes sign")
axes[1].legend(fontsize=8)

lam = np.logspace(-1, 1, 200)                      # wavelength [km]
axes[2].loglog(lam, efolding_time(lam, -0.02), color=BLUE)
for l_km in (0.5, 1.0, 5.0):
    tau = efolding_time(l_km, -0.02)
    axes[2].plot(l_km, tau, "o", color=BLUE, ms=5)
    axes[2].annotate(f"{l_km:g} km: {tau:.2g} yr", (l_km, tau), textcoords="offset points",
                     xytext=(8, -10), fontsize=8)
axes[2].set(xlabel="wavelength (km)", ylabel="e-folding time (yr)",
            title="(c) unstable coast (|D0| = 0.02 km^2/yr):\nshort waves grow fastest")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch07_flux_laws.png", dpi=200)
plt.show()

### Why a bump grows under high-angle waves

On the updrift flank of a seaward bump the shoreline turns toward the waves (smaller $\alpha$); on the downdrift flank it turns away (larger $\alpha$).
Low-angle waves ($dQ_s/d\alpha > 0$): the flux increases across the tip, diverges, and the tip erodes. High-angle waves ($dQ_s/d\alpha < 0$): the flux decreases across the tip, converges, and the tip grows.

*My notes:*

## 7.3 Linear stability and the Fourier view

Normal modes $y_s = \epsilon\,e^{ikx+\sigma t}$ obey $\sigma = -\mathcal D_0k^2$. For $\mathcal D_0 > 0$ every mode decays; for $\mathcal D_0 < 0$ every mode grows,
the shortest fastest: the backward heat equation, ill-posed as a continuum model. A Gaussian bump spreads like the heat kernel of
Chapter 4 when $\mathcal D_0 > 0$, but runs backwards toward a delta spike when $\mathcal D_0 < 0$.

*My notes:*

In [ ]:
D_sf, K = 10.0, 4e5                          # shoreface depth [m]; K [m^3/yr] (Q_s,max = 200,000 m^3/yr)
D_low = shoreline_diffusivity(cerc_flux_derivative(np.radians(20), K), D_sf)    # waves at 20 deg
D_high = shoreline_diffusivity(cerc_flux_derivative(np.radians(60), K), D_sf)   # waves at 60 deg
print(f"D0 = {D_low:,.0f} m^2/yr at 20 deg; {D_high:,.0f} m^2/yr at 60 deg")

xg = np.linspace(-2000, 2000, 801)           # alongshore distance [m]
s0 = 300.0                                    # initial width of the bump [m]
V = 20 * np.sqrt(2 * np.pi) * s0              # plan area of a bump 20 m high
t_star = blowup_time(s0, D_high)

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
for t_yr, color in zip((0, 1.125, 2.25, 4.5), RAMP4):
    axes[0, 0].plot(xg / 1e3, gaussian_bump(xg, t_yr, V, s0, D_low), color=color, label=f"t = {t_yr:g} yr")
axes[0, 0].set(ylim=(0, 95), xlabel="alongshore distance (km)", ylabel="shoreline position (m)",
               title="(a) low-angle waves: the bump spreads (Chapter 4's kernel)")
axes[0, 0].legend(fontsize=8)
for frac, color in zip((0, 0.5, 0.8, 0.95), RAMP4):
    axes[0, 1].plot(xg / 1e3, gaussian_bump(xg, frac * t_star, V, s0, D_high), color=color,
                    label=f"t = {frac * t_star:.2f} yr")
axes[0, 1].set(ylim=(0, 95), xlabel="alongshore distance (km)",
               title=f"(b) high-angle waves: it sharpens toward a spike at t* = {t_star:.2f} yr")
axes[0, 1].legend(fontsize=8)

# The nonlinear one-line model: the same bump plus 5 cm of random noise, invisible at t = 0
dx = 50.0
xs = np.arange(200) * dx                       # a 10 km periodic coast
rng = np.random.default_rng(7)
y0 = 20 * np.exp(-0.5 * ((xs - 5000) / s0) ** 2) + rng.normal(0, 0.05, xs.size)
for ax, a0, name in ((axes[1, 0], 20, "(c) low-angle waves"), (axes[1, 1], 60, "(d) high-angle waves")):
    times, snaps = run_one_line(y0, dx, 0.002, 100, np.radians(a0), D_sf, lambda al: cerc_flux(al, K),
                                save_every=50)
    for t_yr, ys, color in zip(times, snaps, RAMP):
        ax.plot((xs - 5000) / 1e3, ys, color=color, label=f"t = {t_yr:.1f} yr")
    ax.set(xlim=(-2, 2), ylim=(-30, 45), xlabel="alongshore distance (km)",
           title=f"{name}: model with 5 cm of noise")
    ax.legend(fontsize=8, loc="upper left")
axes[1, 0].set_ylabel("shoreline position (m)")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch07_running_backwards.png", dpi=200)
plt.show()

In [ ]:
# Check the numerical model against linear theory: a 2 km sinusoid under low-angle waves
wavelength, eps, n = 2000.0, 0.01, 64
k = 2 * np.pi / wavelength
dxs = wavelength / n
xx = np.arange(n) * dxs
t_end = 1 / (D_low * k**2)                      # one e-folding time
y_end = run_one_line(eps * np.sin(k * xx), dxs, t_end / 2000, 2000, np.radians(20), D_sf,
                     lambda al: cerc_flux(al, K))
print(f"amplitude after one e-folding time: {2 * np.mean(y_end * np.sin(k * xx)) / eps:.4f} "
      f"(linear theory: {np.exp(-1):.4f})")

### Research connection: emergent coastline shapes

Real coasts do not blow up at the grid scale. At finite amplitude, protruding features shadow parts of the coast from the waves
and the nonlinear angular dependence limits growth; cuspate capes, flying spits and alongshore sand waves self-organise
(Ashton et al. 2001; Ashton and Murray 2006). A few robust rules (conservation plus a flux law with a maximum) explain emergent
patterns: the case for reduced-complexity models (Murray 2007).

*My notes:*

## 7.4 A more realistic flux law

With refraction and shoaling, $Q_s \propto H_0^{12/5}\,T^{1/5}\cos^{6/5}\!\alpha\,\sin\alpha$, so $\dfrac{dQ_s}{d\alpha} \propto \cos^{1/5}\!\alpha\left(\cos^2\alpha - \tfrac65\sin^2\alpha\right)$,
which changes sign at $\tan\alpha = \sqrt{5/6}$, i.e. $\alpha_c \approx 42.4^\circ$.

*My notes:*

In [ ]:
al = sp.symbols("alpha", positive=True)
laws = {"sin*cos": sp.sin(al) * sp.cos(al), "sin*cos^(6/5)": sp.sin(al) * sp.cos(al) ** sp.Rational(6, 5)}
for name, Qa in laws.items():
    crit = sp.nsolve(sp.diff(Qa, al), al, 0.7)
    print(f"{name:14s}: D changes sign at {float(sp.deg(crit)):.2f} deg")
print("exact:", sp.atan(sp.sqrt(sp.Rational(5, 6))), "=", f"{float(sp.deg(sp.atan(sp.sqrt(sp.Rational(5, 6))))):.2f} deg")

## Your own system (Exercise 7.4)

| Question | My answer |
|---|---|
| What is conserved? | |
| What is its flux? | |
| On which local gradient or orientation does the flux depend? | |
| Effective diffusivity or wave speed (chain rule) | |
| Can it change sign? When? | |
| What limits growth at small scales and selects a pattern size? | |

*My notes:*

## Exercises 7.1-7.4

*Work space: solve each exercise here, then check against Appendix C of the study guide.*